# Draw a stratified sample for a labeling study

A labeling study asks annotators to judge candidates against a definition the
researcher chooses. The sample decides which candidates they see, so it must
be reproducible: anyone who has the rows, the strata, the seed, and the number
per stratum must get the same candidates.

`draw(table, stratum, n, seed)` ranks the rows of each stratum by the digest of
`sha256(f"{seed}:{stratum}:{word}")`, in ascending order, and takes the first
`n` of each. It returns the sample and a manifest that records what fixes the
sample. This notebook draws one from the fixtures and saves the manifest, the
ids, and an annotation sheet. It never prints a candidate.

In [ ]:
import hashlib
import json
import os
import sys
from pathlib import Path

# kit/ holds the ceolkit package, and this notebook sits in kit/notebooks/
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from ceolkit import draw, load, rank_key, strata

# The fixtures are a sample of version 1.0's rows (kit/fixtures/). To use a
# full config, download its file and point SOURCE at it; with SOURCE = None,
# load() reads the hub, if the dataset is public or you are signed in to it.
SOURCE = Path(os.environ.get("CEOL_FIXTURES", "../fixtures"))

# Where the files are written. Set CEOL_OUT_DIR to change it.
OUT_DIR = Path(os.environ.get("CEOL_OUT_DIR", "labeling_sample"))

## 1. The frame

The frame is the set of rows the study samples from. Choosing it is part of the
design and belongs in the preregistration. This example takes the single-token
rows.

In [ ]:
universe = load("universe", "v1.0", source=SOURCE)
rows = universe.to_pandas()
frame = rows[rows["token_count"] == 1].reset_index(drop=True)
print(f"{len(frame)} of {len(rows)} rows are in the frame")

## 2. The strata

A stratum is a group the sample treats separately: it gives each group the same
number of candidates, so a small group is not lost among a large one. This
example uses the evidence tier. Other ready strata are `spelling_class`,
`length_band`, `frequency_band`, `intake_layer`, and `model_or_engine`. Any
column name, or any function of a row, also works.

In [ ]:
print(strata.evidence_tier.definition)

In [ ]:
sizes = pd.Series(strata.evidence_tier.label_frame(frame)).value_counts()
sizes.rename("rows in the frame")

## 3. The seed and the number per stratum

The seed is a public string. Choose it, and the number per stratum, before you
draw, and write both in the preregistration. Do not try several seeds and keep
the sample you like. The seed below is an example, so replace it.

In [ ]:
SEED = "replace-with-your-own-seed"
N_PER_STRATUM = 20

## 4. Draw

`source` is recorded in the manifest as given. Record the dataset version, the
config, and the sha256 of the file you read, and compare it with the card's
table of files (§4).

In [ ]:
source = {
    "dataset": "ryanjosephkamp/ceol-universe",
    "version": "v1.0",
    "config": "universe",
    "file": "universe_sample.parquet (fixtures)",
    "file_sha256": hashlib.sha256((SOURCE / "universe_sample.parquet").read_bytes()).hexdigest(),
    "frame": "token_count == 1",
}
sample, manifest = draw(frame, strata.evidence_tier, N_PER_STRATUM, SEED, source=source)

print(f"{len(sample)} candidates drawn")
print("sha256 of the sorted ids:", manifest["selected_ids_sha256"])
pd.DataFrame(manifest["strata"]).T

`available` is the stratum's rows in the frame and `selected` is the rows drawn.
A stratum with fewer rows than `n` gives all of them and is listed in
`short_strata`. On the fixtures, `available` counts the fixtures' rows. On the
full config it counts the release's.

In [ ]:
print("strata with fewer rows than asked for:", manifest["short_strata"])

## 5. Check that the draw reproduces

The same rows in any order give the same sample. A second researcher can
confirm this with the manifest alone.

In [ ]:
shuffled = frame.sample(frac=1, random_state=0).reset_index(drop=True)
again, again_manifest = draw(shuffled, strata.evidence_tier, N_PER_STRATUM, SEED, source=source)

assert again["id"].tolist() == sample["id"].tolist()
assert again_manifest == manifest
print("same sample from the shuffled rows:", again_manifest["selected_ids_sha256"] == manifest["selected_ids_sha256"])

## 6. Weights

An equal number from each stratum is not a sample of the frame. To estimate a
share for the whole frame, weight each sampled row by the stratum's rows in the
frame divided by the rows drawn from it.

In [ ]:
weights = pd.Series({label: counts["available"] / counts["selected"]
                     for label, counts in manifest["strata"].items() if counts["selected"]})
weights.rename("frame rows represented by each sampled row")

## 7. Save the manifest, the ids, and the annotation sheet

- `manifest.json` goes with the preregistration.
- `sample_ids.csv` holds ids, strata, and ranks and no candidate, so it can be
  shared and checked against the manifest.
- `annotation_sheet.csv` holds the candidate and empty columns for the
  annotator. It leaves out the stratum and the evidence columns, because an
  annotator who sees them is no longer judging the candidate alone
  (`CODEBOOK.md`). Its row order is a shuffle of the sample by a second seed,
  which uses the same ranking rule. Give each annotator their own order seed.

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

(OUT_DIR / "manifest.json").write_text(
    json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8")
sample[["id", "sample_stratum", "sample_rank"]].to_csv(OUT_DIR / "sample_ids.csv", index=False)

order_seed = SEED + "-order-annotator-1"
order = sorted(range(len(sample)),
               key=lambda i: rank_key(order_seed, "annotation-order", sample["word"].iloc[i]))
sheet = sample.iloc[order][["id", "word"]].assign(
    label="", evidence_type="", evidence_ref="", note="")
sheet.to_csv(OUT_DIR / "annotation_sheet.csv", index=False)

for path in sorted(OUT_DIR.iterdir()):
    print(f"{path.name}: {path.stat().st_size} bytes")

Reading the saved files back and checking them against the manifest:

In [ ]:
saved = json.loads((OUT_DIR / "manifest.json").read_text(encoding="utf-8"))
ids = pd.read_csv(OUT_DIR / "sample_ids.csv", dtype=str)["id"]
digest = hashlib.sha256("\n".join(sorted(ids)).encode("utf-8")).hexdigest()

assert digest == saved["selected_ids_sha256"]
print("the saved ids match the manifest:", digest == saved["selected_ids_sha256"])